# TDA Mapper — Búsqueda de Configuraciones
## Segmentación de Mujeres Embarazadas: Exposición a Ácido Fólico y Riesgo Neonatal

Este notebook extiende el análisis original con una exploración exhaustiva de lentes, parámetros de cubierta y métodos de clustering. Al final se genera una tabla comparativa y se selecciona la mejor configuración.

# DEPENDENCIAS
Instala las dependencias si es necesario:
```
!pip install kmapper umap-learn openpyxl -q
```


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
import networkx as nx
import umap
import kmapper as km


from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import DBSCAN, KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Ruta local del CSV
FILE_NAME = r"Data\Ingesta_AF_clean.csv"
df = pd.read_csv(FILE_NAME)
print("Shape del dataset:", df.shape)

In [ ]:
selected_cols = [
    "uf-af",
    "N° PANES",

    "Marraqueta",
    "Hallulla",
    "Pan molde integral",

    "Antes del embarazo",
    "Durante todo el embarazo",

    "1-3 meses",
    "4-6 meses",
    "7-9 meses",

    "NO consumio",
    "pnacer",
    "Edad_gestacion ultimo hijo",
    "Preeclamsia",
    "Diabetes gestacional",
    "Embarazo multiple",
    "edad",
    "neduc"
]

data = df[selected_cols].copy()
for col in data.columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")

# Variables derivadas
data["prematuro"]         = (data["Edad_gestacion ultimo hijo"] < 37).astype(int)
data["bajo_peso"]         = (data["pnacer"] < 2500).astype(int)
data["riesgo_obstetrico"] = (
    data["Preeclamsia"].fillna(0)
    + data["Diabetes gestacional"].fillna(0)
    + data["Embarazo multiple"].fillna(0)
)]
data["score_pan"] = (
    data["N° PANES"].fillna(0)
    * (data["Marraqueta"].fillna(0)
       + data["Hallulla"].fillna(0)
       + data["Pan molde integral"].fillna(0))
)]
data["AF_exposure"] = (
    data["uf-af"].fillna(0)
    + data["score_pan"].fillna(0) * 100
)]

tda_features = [
    "AF_exposure",
    "uf-af",
    "score_pan",

    "Antes del embarazo",
    "Durante todo el embarazo",
    "1-3 meses",
    "4-6 meses",
    "7-9 meses",

    "riesgo_obstetrico",

    "edad",
    "neduc"
]

X = data[tda_features].copy()

# Imputación y escalado
imputer  = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print("Shape de X_scaled:", X_scaled.shape)
print("Nulos tras imputación:", pd.DataFrame(X_scaled).isnull().sum().sum())

In [ ]:
def compute_graph_metrics(graph, X_scaled):
    nodes = graph["nodes"]
    links = graph["links"]

    n_nodes = len(nodes)
    n_links = sum(len(v) for v in links.values())

    if n_nodes == 0:
        return {"score_compuesto": -999, "n_nodes": 0, "n_links": 0, "n_components": 0,
                "avg_node_size": 0, "pct_covered": 0, "pct_isolated": 100,
                "silhouette": np.nan, "calinski": np.nan, "davies_bouldin": np.nan}

    node_sizes = [len(v) for v in nodes.values()]
    avg_size = np.mean(node_sizes)

    all_points = set()
    for pts in nodes.values():
        all_points.update(pts)
    pct_covered = len(all_points) / X_scaled.shape[0] * 100
    pct_isolated = 100 - pct_covered

    G = nx.Graph()
    G.add_nodes_from(nodes.keys())
    for src, tgts in links.items():
        for tgt in tgts:
            G.add_edge(src, tgt)
    n_components = nx.number_connected_components(G)

    labels = np.full(X_scaled.shape[0], -1)
    for i, (node_id, pts) in enumerate(nodes.items()):
        for pt in pts:
            if labels[pt] == -1:
                labels[pt] = i

    valid_mask = labels >= 0
    unique_labels = np.unique(labels[valid_mask])

    silhouette = np.nan
    calinski = np.nan
    davies = np.nan

    if len(unique_labels) >= 2 and valid_mask.sum() > len(unique_labels):
        try:
            silhouette = silhouette_score(X_scaled[valid_mask], labels[valid_mask], sample_size=min(1000, valid_mask.sum()), random_state=RANDOM_STATE)
            calinski = calinski_harabasz_score(X_scaled[valid_mask], labels[valid_mask])
            davies = davies_bouldin_score(X_scaled[valid_mask], labels[valid_mask])
        except Exception:
            pass

    trivial_penalty = 0
    if n_nodes < 5:
        trivial_penalty += 3
    if n_components <= 1:
        trivial_penalty += 4
    if pct_covered < 60:
        trivial_penalty += 2

    score = (
        1.5 * np.log1p(n_nodes)
        + 1.0 * np.log1p(n_links)
        + 1.0 * (pct_covered / 100)
        + 0.5 * (silhouette if not np.isnan(silhouette) else -0.5)
        - 0.3 * (davies if not np.isnan(davies) else 2.0)
        - 1.5 * n_components
        - trivial_penalty
    )

    return {
        "n_nodes": n_nodes,
        "n_links": n_links,
        "n_components": n_components,
        "avg_node_size": round(avg_size, 1),
        "pct_covered": round(pct_covered, 1),
        "pct_isolated": round(pct_isolated, 1),
        "silhouette": round(silhouette, 4) if not np.isnan(silhouette) else np.nan,
        "calinski": round(calinski, 2) if not np.isnan(calinski) else np.nan,
        "davies_bouldin": round(davies, 4) if not np.isnan(davies) else np.nan,
        "score_compuesto": round(score, 4)
    }

In [ ]:
LENS_CONFIGS = [
    {"lens": "PCA_1", "lens_params": {}},
    {"lens": "PCA_2", "lens_params": {}},
    {"lens": "TSNE",  "lens_params": {"perplexity": 15}},
    {"lens": "TSNE",  "lens_params": {"perplexity": 30}},
    {"lens": "TSNE",  "lens_params": {"perplexity": 50}},
    {"lens": "UMAP",  "lens_params": {"n_neighbors": 10, "min_dist": 0.05}},
    {"lens": "UMAP",  "lens_params": {"n_neighbors": 15, "min_dist": 0.10}},
    {"lens": "UMAP",  "lens_params": {"n_neighbors": 30, "min_dist": 0.20}},
]

COVER_CONFIGS = [
    {"n_cubes": 8,  "perc_overlap": 0.35},
    {"n_cubes": 12, "perc_overlap": 0.45},
    {"n_cubes": 15, "perc_overlap": 0.50},
    {"n_cubes": 20, "perc_overlap": 0.40},
]

CLUSTER_CONFIGS = [
    {"clusterer": "DBSCAN",        "cluster_params": {"eps": 0.5,  "min_samples": 4}},
    {"clusterer": "DBSCAN",        "cluster_params": {"eps": 0.9,  "min_samples": 6}},
    {"clusterer": "DBSCAN",        "cluster_params": {"eps": 1.2,  "min_samples": 4}},
    {"clusterer": "KMeans",        "cluster_params": {"n_clusters": 3}},
    {"clusterer": "KMeans",        "cluster_params": {"n_clusters": 5}},
    {"clusterer": "Agglomerative", "cluster_params": {"n_clusters": 4, "linkage": "ward"}},
    {"clusterer": "Agglomerative", "cluster_params": {"n_clusters": 6, "linkage": "average"}},
    {"clusterer": "GaussianMixture","cluster_params": {"n_components": 4}},
]

total_exp = len(LENS_CONFIGS) * len(COVER_CONFIGS) * len(CLUSTER_CONFIGS)
print(f"Total de configuraciones a evaluar: {total_exp}")
print(f"Lentes: {len(LENS_CONFIGS)} | Coberturas: {len(COVER_CONFIGS)} | Clusterers: {len(CLUSTER_CONFIGS)}")

def build_lens(X_scaled, lens_name, params):
    try:
        if lens_name == "PCA_1":
            proj = PCA(n_components=1, random_state=RANDOM_STATE).fit_transform(X_scaled)
        elif lens_name == "PCA_2":
            proj = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_scaled)
        elif lens_name == "TSNE":
            proj = TSNE(
                n_components=2,
                perplexity=params.get("perplexity", 30),
                random_state=RANDOM_STATE,
                n_iter=500
            ).fit_transform(X_scaled)
        elif lens_name == "UMAP":
            proj = umap.UMAP(
                n_components=2,
                n_neighbors=params.get("n_neighbors", 15),
                min_dist=params.get("min_dist", 0.1),
                random_state=RANDOM_STATE
            ).fit_transform(X_scaled)
        else:
            return None
        return proj
    except Exception as e:
        print(f"  ⚠ Error en lente {lens_name}: {e}")
        return None

In [ ]:
def build_clusterer(cluster_name, params):
    if cluster_name == "DBSCAN":
        return DBSCAN(eps=params.get("eps", 0.9), min_samples=params.get("min_samples", 6))
    elif cluster_name == "KMeans":
        return KMeans(n_clusters=params.get("n_clusters", 5), random_state=RANDOM_STATE, n_init=10)
    elif cluster_name == "Agglomerative":
        return AgglomerativeClustering(n_clusters=params.get("n_clusters", 5), linkage=params.get("linkage", "ward"))
    elif cluster_name == "GaussianMixture":
        return GaussianMixture(n_components=params.get("n_components", 5), random_state=RANDOM_STATE)
    else:
        return DBSCAN()

lens_cache = {}
for lc in LENS_CONFIGS:
    key = (lc["lens"], str(sorted(lc["lens_params"].items())))
    if key not in lens_cache:
        t0 = time.time()
        proj = build_lens(X_scaled, lc["lens"], lc["lens_params"])
        elapsed = time.time() - t0
        lens_cache[key] = proj
        if proj is not None:
            print(f"{lc['lens']} {lc['lens_params']} → shape {proj.shape}  ({elapsed:.1f}s)")
        else:
            print(f"{lc['lens']} {lc['lens_params']} falló")

print(f"\n{len(lens_cache)} proyecciones en caché")

In [ ]:
resultados = []
mapper_obj = km.KeplerMapper(verbose=0)

exp_id = 0
for lc in LENS_CONFIGS:
    key = (lc["lens"], str(sorted(lc["lens_params"].items())))
    lens_proj = lens_cache.get(key)
    if lens_proj is None:
        continue

    for cc in COVER_CONFIGS:
        cover = km.Cover(n_cubes=cc["n_cubes"], perc_overlap=cc["perc_overlap"])

        for cl in CLUSTER_CONFIGS:
            exp_id += 1
            clusterer = build_clusterer(cl["clusterer"], cl["cluster_params"])

            try:
                t0 = time.time()
                graph = mapper_obj.map(
                    lens_proj, X_scaled,
                    cover=cover,
                    clusterer=clusterer
                )
                elapsed = time.time() - t0
                metrics = compute_graph_metrics(graph, X_scaled)
            except Exception as e:
                print(f"Experimento {exp_id} falló: {e}")
                metrics = {"score_compuesto": -99}
                elapsed = 0

            row = {
                "exp_id":       exp_id,
                "lente":        lc["lens"],
                "lens_params":  str(lc["lens_params"]) if lc["lens_params"] else "",
                "n_cubes":      cc["n_cubes"],
                "perc_overlap": cc["perc_overlap"],
                "clusterer":    cl["clusterer"],
                "cluster_params": str(cl["cluster_params"]),
                "tiempo_s":     round(elapsed, 2),
                **metrics
            }
            resultados.append(row)

df_results = pd.DataFrame(resultados)
print(f\n{len(df_results)} experimentos completados
,
,
score_compuesto", ascending=False).reset_index(drop=True)

COLS_DISPLAY = [
    "exp_id", "lente", "lens_params", "n_cubes", "perc_overlap",
    "clusterer", "cluster_params",
    "n_nodes", "n_links", "n_components",
    "avg_node_size", "pct_covered", "pct_isolated",
    "silhouette", "davies_bouldin", "score_compuesto"
]

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)
display(df_sorted[COLS_DISPLAY].head(20))

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("Exploración de Configuraciones Mapper\nEje: métricas por tipo de lente",
             fontsize=15, fontweight="bold")

palette = {"PCA_1": "#4C72B0", "PCA_2": "#DD8452",
           "TSNE":  "#55A868", "UMAP":  "#C44E52"}

# (plots...)
ax = axes[0, 0]
df_results.boxplot(column="score_compuesto", by="lente", ax=ax,
                   boxprops=dict(color="steelblue"),
                   medianprops=dict(color="red", linewidth=2))
ax.set_title("Score Compuesto por Lente")
ax.set_xlabel("Lente"); ax.set_ylabel("Score")
plt.sca(ax); plt.xticks(rotation=20)

ax = axes[0, 1]
df_results.dropna(subset=["silhouette"]).boxplot(
    column="silhouette", by="clusterer", ax=ax,
    boxprops=dict(color="darkorange"),
    medianprops=dict(color="red", linewidth=2))
ax.set_title("Silhouette por Clusterer")
ax.set_xlabel("Clusterer"); ax.set_ylabel("Silhouette")
plt.sca(ax); plt.xticks(rotation=20)

ax = axes[0, 2]
for lente, grp in df_results.groupby("lente"):
    ax.scatter(grp["n_nodes"], grp["n_links"],
               label=lente, alpha=0.6,
               color=palette.get(lente, "gray"), s=40)
ax.set_title("Nodos vs Links por Lente")
ax.set_xlabel("N° Nodos"); ax.set_ylabel("N° Links")
ax.legend(fontsize=8)

ax = axes[1, 0]
df_results.boxplot(column="pct_covered", by="lente", ax=ax,
                   boxprops=dict(color="seagreen"),
                   medianprops=dict(color="red", linewidth=2))
ax.set_title("% Puntos Cubiertos por Lente")
ax.set_xlabel("Lente"); ax.set_ylabel("% Cubiertos")
plt.sca(ax); plt.xticks(rotation=20)

ax = axes[1, 1]
for nc, grp in df_results.groupby("n_cubes"):
    ax.scatter(grp["n_cubes"] + np.random.uniform(-0.3, 0.3, len(grp)),
               grp["n_components"],
               alpha=0.4, s=25, label=f"{nc} cubos")
ax.set_title("N° Componentes Conectadas vs N° Cubos")
ax.set_xlabel("N° Cubos"); ax.set_ylabel("Componentes")
ax.legend(fontsize=8)

ax = axes[1, 2]
pivot = df_results.pivot_table(
    values="score_compuesto",
    index="lente", columns="clusterer",
    aggfunc="mean"
)
sns.heatmap(pivot, ax=ax, annot=True, fmt=".3f",
            cmap="RdYlGn", linewidths=0.5,
            annot_kws={"size": 9})
ax.set_title("Score Compuesto Promedio\n(Lente × Clusterer)")
ax.set_xlabel("Clusterer"); ax.set_ylabel("Lente")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("mapper_comparacion.png", dpi=150, bbox_inches="tight")
plt.show()

best = df_sorted.iloc[0] if not df_sorted.empty else None

if best is not None:
    print("═" * 70)
    print("MEJOR CONFIGURACIÓN")
    print("═" * 70)
    print(f"  Lente:            {best['lente']}  {best['lens_params']}")
    print(f"  Cubierta:         {best['n_cubes']} cubos / {best['perc_overlap']*100:.0f}% overlap")
    print(f"  Clusterer:        {best['clusterer']}  {best['cluster_params']}")
    print("─" * 70)
    print(f"  Nodos:            {best['n_nodes']}")
    print(f"  Links:            {best['n_links']}")
    print(f"  Componentes:      {best['n_components']}")
    print(f"  Tamaño prom nodo: {best['avg_node_size']}")
    print(f"  % Cubiertos:      {best['pct_covered']}%")
    print(f"  Silhouette:       {best['silhouette']}")
    print(f"  Davies-Bouldin:   {best['davies_bouldin']}")
    print(f"  Score compuesto:  {best['score_compuesto']}")
else:
    print("No se encontraron resultados.")